In [1]:
#Bloco 1 — Importar bibliotecas e carregar os dados
import pandas as pd
import numpy as np
from scipy.stats import gamma, norm

# Carrega a clustering table
df = pd.read_csv('../outputs/clustering_table.csv')

print(f"Shape: {df.shape}")
print(df['policy_cluster'].value_counts())

Shape: (1388, 16)
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64


In [2]:
# Bloco 2 — Definir o rules engine
cluster_rules = {
    'MTS': {
        'model': 'EOQ-ROP',
        'z': 1.65,        # 95% nível de serviço
    },
    'Comakership': {
        'model': 'EOQ-ROP',
        'z': 1.28,        # 90% nível de serviço
    },
    'Batch-to-Order': {
        'model': 'EOQ-ROP',
        'z': 1.28,
    },
    'Dynamic': {
        'model': '(s,S)',
        'z': 1.28,
    },
    'MTO': {
        'model': None,
        'z': 0,
    },
}

print("Rules engine definido:")
for cluster, rules in cluster_rules.items():
    print(f"  {cluster}: modelo={rules['model']}, z={rules['z']}")

Rules engine definido:
  MTS: modelo=EOQ-ROP, z=1.65
  Comakership: modelo=EOQ-ROP, z=1.28
  Batch-to-Order: modelo=EOQ-ROP, z=1.28
  Dynamic: modelo=(s,S), z=1.28
  MTO: modelo=None, z=0


In [18]:
#Bloco 3a — Custo de holding a partir do Alloy_Brass_price.xlsx
# Carrega preços do latão
preco_df = pd.read_excel('../data/Alloy_Brass_price.xlsx', 
                          sheet_name='Quotazioni')
preco_df['Data'] = pd.to_datetime(preco_df['Data'], dayfirst=True)

# Filtra período 2023-2024
preco_2324 = preco_df[
    (preco_df['Data'] >= '2023-01-01') & 
    (preco_df['Data'] <= '2024-12-31')
]

# Preço médio em €/kg
preco_cw614_kg = preco_2324['Quotazione CW614N/CW617N'].mean() / 1000
preco_cw724_kg = preco_2324['Quotazione CW724R'].mean() / 1000

print(f"Preço médio CW614N: €{preco_cw614_kg:.3f}/kg")
print(f"Preço médio CW724R: €{preco_cw724_kg:.3f}/kg")

Preço médio CW614N: €6.978/kg
Preço médio CW724R: €8.996/kg


c:\Users\italo\anaconda3\envs\dt_ai\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [19]:
#Bloco 3b — Lead time a partir do Variability.xlsx
# Carrega custos
costs = pd.read_excel('../data/Variability.xlsx', sheet_name='Costs')
costs.columns = ['Description', 'Value', 'Unit']
costs = costs.dropna(subset=['Description'])

setup_cost_drawing   = costs[costs['Description'].str.contains('drawing',   case=False)]['Value'].values[0]
setup_cost_extrusion = costs[costs['Description'].str.contains('extrusion', case=False)]['Value'].values[0]
holding_cost_rate    = costs[costs['Description'].str.contains('holding',   case=False)]['Value'].values[0]
raw_material_cost    = costs[costs['Description'].str.contains('raw material', case=False)]['Value'].values[0]

# Carrega OEE por máquina
prod_var = pd.read_excel('../data/Variability.xlsx', 
                          sheet_name='Production variability')

# Extrai média e desvio padrão do OEE por grupo
oee_media = prod_var[['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 7']].copy()
oee_media.columns = ['group', 'machine_code', 'avg_OEE']
oee_media = oee_media.dropna(subset=['avg_OEE'])
oee_media = oee_media[oee_media['avg_OEE'] != 'OEE']
oee_media['avg_OEE'] = pd.to_numeric(oee_media['avg_OEE'], errors='coerce')
oee_media = oee_media.dropna(subset=['avg_OEE'])

oee_std = prod_var[['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 11']].copy()
oee_std.columns = ['group', 'machine_code', 'std_OEE']
oee_std = oee_std.dropna(subset=['std_OEE'])
oee_std = oee_std[oee_std['std_OEE'] != 'OEE']
oee_std['std_OEE'] = pd.to_numeric(oee_std['std_OEE'], errors='coerce')
oee_std = oee_std.dropna(subset=['std_OEE'])

# Lead time nominal por estágio em dias
LT_nominal = {
    'FONDERIA':   2.0,
    'ESTRUSIONE': 1.0,
    'FINITURA':   1.5,
}

# Calcula LT efetivo e σ_LT por grupo
lt_efetivo = {}
sigma_lt   = {}

for group in LT_nominal:
    oee_m = oee_media[oee_media['group'] == group]['avg_OEE'].mean()
    oee_s = oee_std[oee_std['group'] == group]['std_OEE'].mean()
    lt_n  = LT_nominal[group]
    
    lt_efetivo[group] = lt_n / oee_m
    sigma_lt[group]   = lt_n * oee_s / (oee_m ** 2)

# Lead time total em meses
LT       = sum(lt_efetivo.values()) / 30
sigma_LT = np.sqrt(sum(v**2 for v in sigma_lt.values())) / 30

print(f"LT total:  {LT:.4f} meses ({LT*30:.2f} dias)")
print(f"σ_LT:      {sigma_LT:.4f} meses ({sigma_LT*30:.2f} dias)")

LT total:  0.2553 meses (7.66 dias)
σ_LT:      0.0280 meses (0.84 dias)


In [16]:
# Lead time nominal por estágio em dias (tempo teórico sem perdas)
LT_nominal = {
    'FONDERIA':   2.0,  # fundição
    'ESTRUSIONE': 1.0,  # extrusão
    'FINITURA':   1.5,  # acabamento
}

# Carrega grupo de cada máquina da aba Production variability
prod_var_clean = pd.read_excel('../data/Variability.xlsx', 
                                sheet_name='Production variability')

# Extrai grupo, código e OEE médio
grupos = prod_var_clean[['Unnamed: 1', 'Unnamed: 2', 
                          'Unnamed: 7']].copy()
grupos.columns = ['group', 'machine_code', 'avg_OEE']
grupos = grupos.dropna(subset=['avg_OEE'])
grupos = grupos[grupos['avg_OEE'] != 'OEE']
grupos['avg_OEE'] = pd.to_numeric(grupos['avg_OEE'], errors='coerce')
grupos = grupos.dropna(subset=['avg_OEE'])

# Calcula LT efetivo por grupo
# LT_efetivo = LT_nominal / OEE_medio
lt_por_grupo = {}
for group, data in grupos.groupby('group'):
    oee_medio = data['avg_OEE'].mean()
    lt_nominal = LT_nominal.get(group, 1.0)
    lt_efetivo = lt_nominal / oee_medio
    lt_por_grupo[group] = round(lt_efetivo, 2)

print("Lead time efetivo por grupo (dias):")
for group, lt in lt_por_grupo.items():
    print(f"  {group}: {lt} dias")

# Lead time total médio passando pelos 3 estágios
LT_total_dias = sum(lt_por_grupo.values())
LT_total_meses = LT_total_dias / 30

print(f"\nLT total: {LT_total_dias:.2f} dias → {LT_total_meses:.4f} meses")

# Extrai desvio padrão do OEE por grupo
std_oee_grupo = {}
prod_var_std = prod_var_clean[['Unnamed: 1', 'Unnamed: 2', 
                                'Unnamed: 11']].copy()
prod_var_std.columns = ['group', 'machine_code', 'std_OEE']
prod_var_std = prod_var_std.dropna(subset=['std_OEE'])
prod_var_std = prod_var_std[prod_var_std['std_OEE'] != 'OEE']
prod_var_std['std_OEE'] = pd.to_numeric(prod_var_std['std_OEE'], errors='coerce')
prod_var_std = prod_var_std.dropna(subset=['std_OEE'])

for group, data in prod_var_std.groupby('group'):
    std_oee = data['std_OEE'].mean()
    lt_nominal = LT_nominal.get(group, 1.0)
    # Propagação da variabilidade: σ_LT = LT_nominal × σ_OEE / OEE²
    oee_medio = grupos[grupos['group'] == group]['avg_OEE'].mean()
    sigma_lt = lt_nominal * std_oee / (oee_medio ** 2)
    std_oee_grupo[group] = round(sigma_lt, 4)

print("Desvio padrão do LT por grupo (dias):")
for group, sigma in std_oee_grupo.items():
    print(f"  {group}: {sigma} dias")

# σ_LT total (soma quadrática das variabilidades)
sigma_LT_total_dias = np.sqrt(sum(v**2 for v in std_oee_grupo.values()))
sigma_LT_total_meses = sigma_LT_total_dias / 30

print(f"\nσ_LT total: {sigma_LT_total_dias:.4f} dias → {sigma_LT_total_meses:.4f} meses")

Lead time efetivo por grupo (dias):
  ESTRUSIONE: 1.75 dias
  FINITURA: 3.41 dias
  FONDERIA: 2.5 dias

LT total: 7.66 dias → 0.2553 meses
Desvio padrão do LT por grupo (dias):
  ESTRUSIONE: 0.3866 dias
  FINITURA: 0.6346 dias
  FONDERIA: 0.3938 dias

σ_LT total: 0.8410 dias → 0.0280 meses


In [20]:
# Bloco 3c — Consolidação de todos os parâmetros

# Custo de holding por liga
H_cw614 = preco_cw614_kg * holding_cost_rate
H_cw724 = preco_cw724_kg * holding_cost_rate

# Setup cost — assumindo 1h de setup por ordem
S_drawing   = setup_cost_drawing   * 1.0
S_extrusion = setup_cost_extrusion * 1.0
S_medio     = (S_drawing + S_extrusion) / 2

print("── Parâmetros consolidados ──")
print(f"LT médio:     {LT:.4f} meses  ({LT*30:.2f} dias)")
print(f"σ_LT:         {sigma_LT:.4f} meses  ({sigma_LT*30:.2f} dias)")
print(f"H CW614N:     €{H_cw614:.4f}/kg/ano")
print(f"H CW724R:     €{H_cw724:.4f}/kg/ano")
print(f"S drawing:    €{S_drawing:.0f}/ordem")
print(f"S extrusion:  €{S_extrusion:.0f}/ordem")
print(f"S médio:      €{S_medio:.0f}/ordem")

── Parâmetros consolidados ──
LT médio:     0.2553 meses  (7.66 dias)
σ_LT:         0.0280 meses  (0.84 dias)
H CW614N:     €0.6978/kg/ano
H CW724R:     €0.8996/kg/ano
S drawing:    €80/ordem
S extrusion:  €120/ordem
S médio:      €100/ordem


In [21]:
# Bloco 4 — Calcular SS, ROP e EOQ por item
from scipy.stats import gamma as gamma_dist, norm as norm_dist

params = []

for _, row in df.iterrows():
    cluster = row['policy_cluster']
    rules   = cluster_rules[cluster]
    
    # MTO não recebe parâmetros
    if rules['model'] is None:
        params.append({
            'Item':            row['Item'],
            'policy_cluster':  cluster,
            'inventory_model': None,
            'SS':  0, 'ROP': 0, 'EOQ': 0,
            's':   0, 'S':   0,
        })
        continue
    
    z        = rules['z']
    mu       = row['avg_monthly']      # demanda média mensal
    sigma    = row['std_monthly']      # desvio padrão mensal
    adi      = row['ADI']
    D_annual = row['D_annual']         # demanda anual em kg
    classe   = row['Classe']
    
    # Custo de holding por liga
    H = H_cw724 if classe in [44, 46] else H_cw614
    
    # Setup cost médio
    S = S_medio
    
    # ── Safety Stock ──
    if adi < 1.32:
        # Distribuição Normal
        SS = z * np.sqrt((sigma**2 * LT) + (sigma_LT**2 * mu**2))
    else:
        # Distribuição Gamma
        if mu > 0 and sigma > 0:
            k     = (mu**2) / (sigma**2)
            theta = (sigma**2) / mu
            SS    = gamma_dist.ppf(0.95, a=k, scale=theta) - (mu * LT)
            SS    = max(SS, 0)
        else:
            SS = 0
    
    # ── ROP ──
    ROP = (mu * LT) + SS
    
    # ── EOQ ──
    if H > 0 and D_annual > 0:
        EOQ = np.sqrt((2 * D_annual * S) / H)
    else:
        EOQ = 0
    
    # ── s e S para modelo (s,S) ──
    s_param = ROP
    S_param = ROP + EOQ

    params.append({
        'Item':            row['Item'],
        'policy_cluster':  cluster,
        'inventory_model': rules['model'],
        'SS':  round(SS,  2),
        'ROP': round(ROP, 2),
        'EOQ': round(EOQ, 2),
        's':   round(s_param, 2),
        'S':   round(S_param, 2),
    })

params_df = pd.DataFrame(params)

print(f"Parâmetros calculados para {len(params_df)} itens")
print(params_df['policy_cluster'].value_counts())
print(params_df.head(10))

Parâmetros calculados para 1388 itens
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64
     Item policy_cluster inventory_model       SS      ROP      EOQ        s  \
0  300007            MTO             NaN     0.00     0.00     0.00     0.00   
1  300019        Dynamic           (s,S)  3182.28  3328.96  1405.58  3328.96   
2  300021            MTO             NaN     0.00     0.00     0.00     0.00   
3  300027            MTO             NaN     0.00     0.00     0.00     0.00   
4  300042        Dynamic           (s,S)   940.42   983.80   764.44   983.80   
5  300082        Dynamic           (s,S)   845.04   890.38   781.49   890.38   
6  300091            MTS         EOQ-ROP   550.06   750.03  1641.16   750.03   
7  300092        Dynamic           (s,S)  1312.85  1378.87   942.95  1378.87   
8  300111        Dynamic           (s,S)   522.65   547.93   583.50   547.93   
9  300120        Dynamic           (s,S)  179

In [22]:
#Bloco 5 Verifica itens conhecidos

itens_check = ['300091', '300007', '300092', '300019']

check = params_df[params_df['Item'].astype(str).isin(itens_check)]

print(check.to_string(index=False))

  Item policy_cluster inventory_model      SS     ROP     EOQ       s       S
300007            MTO             NaN    0.00    0.00    0.00    0.00    0.00
300019        Dynamic           (s,S) 3182.28 3328.96 1405.58 3328.96 4734.53
300091            MTS         EOQ-ROP  550.06  750.03 1641.16  750.03 2391.19
300092        Dynamic           (s,S) 1312.85 1378.87  942.95 1378.87 2321.82


In [23]:
#Bloco 6 Salvar parametrização
params_df.to_csv('../outputs/params_table.csv', index=False)
print("Params table salva com sucesso.")

Params table salva com sucesso.
